# Sentiment Darija : TF-IDF normalisé et caractères 3–5

Ce notebook compare une référence mots avec des n-grammes de caractères (3, 4 et 5) et des mots (1 et 2). Le vocabulaire et l'IDF sont appris uniquement sur train. La normalisation L2 et la régularisation limitent l'importance des textes longs et des caractéristiques rares.

Les réglages sont sélectionnés sur **dev macro-F1**. Le test officiel est évalué seulement après cette sélection. Il a déjà été consulté dans les expériences précédentes : les nouveaux scores restent exploratoires et devront être confirmés sur un jeu indépendant.

Exécution depuis PowerShell : `./scripts/Invoke-SparkNotebook.ps1`. Les cellules s'exécutent dans le conteneur Spark avec les deux workers. Aucun fichier Python n'est exporté dans le dépôt.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter
from uuid import uuid4
import csv
import hashlib
import json
import math
import time
import urllib.request

from pyspark import StorageLevel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import (
    CountVectorizer, HashingTF, IDF, NGram, Normalizer,
    RegexTokenizer, SQLTransformer, VectorAssembler,
)
from pyspark.sql import SparkSession, functions as F, types as T

INPUT_DIR = Path("/opt/lahjalens/data/processed/afrisenti/v1")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
LOCAL_DIR = INPUT_DIR.parent / "experiments" / RUN_ID
HDFS_DIR = "hdfs://namenode:9000/data/lahjalens/models/afrisenti_tfidf_v2/" + RUN_ID
LABEL_NAMES = ["negative", "neutral", "positive"]
FIELDS = ["record_id", "comment_text", "sentiment", "label", "source_dataset",
          "source_split", "language", "cleaning_version"]

spark = (SparkSession.builder.appName("LahjaLens-TFIDF-Normalized-" + RUN_ID)
         .master("spark://spark-master:7077")
         .config("spark.hadoop.dfs.replication", "2")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.default.parallelism", "4")
         .config("spark.sql.csv.parser.columnPruning.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
assert spark.sparkContext.master == "spark://spark-master:7077"
spark.sparkContext._jsc.hadoopConfiguration().set("dfs.replication", "2")
LOCAL_DIR.mkdir(parents=True, exist_ok=False)

def write_local(name, data):
    target = LOCAL_DIR / name
    target.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                      encoding="utf-8")

report = {"run_id": RUN_ID, "spark_application_id": spark.sparkContext.applicationId,
          "master": spark.sparkContext.master, "spark_version": spark.version,
          "dataset": "afrisenti_ary", "selection_metric": "dev.macro_f1",
          "historical_test_already_consulted": True, "results": [], "status": "running"}
print("RUN_ID", RUN_ID, flush=True)


## 1. Vérifier chaque enregistrement avant l'apprentissage

On compare les colonnes lues par Spark avec le lecteur CSV standard, ainsi qu'avec les empreintes et les effectifs de l'audit. Les guillemets doublés et les champs multilignes sont explicitement configurés. Une ligne invalide provoque une erreur ; elle n'est pas supprimée silencieusement.

L'ancien lecteur est reproduit uniquement pour expliquer les écarts. Les identifiants récupérés figurent dans le rapport.


In [ ]:
audit = json.loads((INPUT_DIR / "audit.json").read_text(encoding="utf-8"))
schema = T.StructType([T.StructField(name, T.StringType(), True) for name in FIELDS])
frames, reference, input_audit = {}, {}, {}

for split in ("train", "dev", "test"):
    path = INPUT_DIR / (split + ".csv")
    payload = path.read_bytes()
    assert hashlib.sha256(payload).hexdigest() == audit["splits"][split]["sha256"], split
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, strict=True)
        assert reader.fieldnames == FIELDS, reader.fieldnames
        rows = list(reader)
    assert all(None not in row and all(v is not None for v in row.values()) for row in rows)
    assert len(rows) == audit["splits"][split]["rows"]
    expected = {r["record_id"]: r for r in rows}
    assert len(expected) == len(rows), "Identifiants dupliqués"
    reference[split] = expected

    frame = (spark.read.schema(schema).options(
        header=True, enforceSchema=False, mode="FAILFAST", encoding="UTF-8",
        quote='"', escape='"', multiLine=True,
    ).csv(str(path)).cache())
    actual = {r["record_id"]: r.asDict() for r in frame.collect()}
    assert actual == expected, "Le CSV Spark diffère du CSV de référence : " + split
    assert all(r["comment_text"].strip() and r["sentiment"] in LABEL_NAMES
               and r["label"] == str(LABEL_NAMES.index(r["sentiment"]))
               and r["source_split"] == split for r in rows)
    counts = dict(Counter(r["sentiment"] for r in rows))
    assert counts == audit["splits"][split]["labels"]
    assert set(counts) == set(LABEL_NAMES)

    legacy = (spark.read.options(header=True, mode="PERMISSIVE").csv(str(path))
              .select("record_id", "comment_text", F.col("label").cast("double").alias("label"))
              .where(F.col("comment_text").isNotNull() & (F.length(F.trim("comment_text")) > 0)
                     & F.col("label").isin(0.0, 1.0, 2.0)))
    legacy_ids = {r["record_id"] for r in legacy.select("record_id").collect()}
    recovered = sorted(set(expected) - legacy_ids)
    input_audit[split] = {
        "rows": len(rows), "labels": counts, "sha256": hashlib.sha256(payload).hexdigest(),
        "legacy_accepted_rows": legacy.count(), "recovered_ids": recovered,
        "recovered_contain_quotes": [bool('"' in expected[key]["comment_text"]) for key in recovered],
    }
    frames[split] = frame.withColumn("label", F.col("label").cast("double"))

text_keys = {split: {r["comment_text"] for r in values.values()} for split, values in reference.items()}
for left, right in (("train", "dev"), ("train", "test"), ("dev", "test")):
    assert not (text_keys[left] & text_keys[right]), "Fuite de texte entre " + left + "/" + right

counts = {LABEL_NAMES.index(name): count for name, count in input_audit["train"]["labels"].items()}
total = sum(counts.values())
weights = {label: total / (3.0 * count) for label, count in counts.items()}
weight_map = F.create_map(*[v for label, weight in sorted(weights.items())
                           for v in (F.lit(float(label)), F.lit(weight))])
train = frames["train"].withColumn("class_weight", weight_map[F.col("label")])
dev = frames["dev"]
report["input_audit"], report["class_weights"] = input_audit, weights
report["csv_read_options"] = {"quote": '"', "escape": '"', "multiLine": True, "mode": "FAILFAST"}
write_local("input_audit.json", input_audit)
write_local("report.json", report)
print("INPUT_AUDIT", json.dumps(input_audit, ensure_ascii=False), flush=True)


## 2. Représentations comparées

- **Référence mots** : HashingTF + IDF + régression logistique, réglage précédent.
- **Caractères 3–5** : vocabulaire appris sur train, TF-IDF puis normalisation L2.
- **Mots 1–2 + caractères 3–5** : deux vecteurs TF-IDF normalisés, assemblés et normalisés.

Les négations, les emojis et les écritures Arabizi sont conservés. La mise en minuscules est la seule transformation additionnelle. Les étapes de création des caractères font partie du modèle sauvegardé ; il pourra donc prédire à partir de la seule colonne comment_text.


In [ ]:
tokenizer = RegexTokenizer(inputCol="comment_text", outputCol="word_tokens",
                           pattern=r"\s+", minTokenLength=1)
probe = spark.createDataFrame([("salam   zwina\tbzaf",)], ["comment_text"])
assert tokenizer.transform(probe).first()["word_tokens"] == ["salam", "zwina", "bzaf"]

lower_text = SQLTransformer(statement="SELECT *, lower(comment_text) AS feature_text FROM __THIS__")
char_tokens = SQLTransformer(statement="""
SELECT *, flatten(transform(array(3, 4, 5), n ->
    CASE WHEN length(feature_text) >= n THEN
        transform(sequence(1, length(feature_text) - n + 1),
                  i -> substring(feature_text, i, n))
    ELSE cast(array() AS array<string>) END
)) AS char_tokens FROM __THIS__
""")
probe_chars = char_tokens.transform(lower_text.transform(
    spark.createDataFrame([("abcde",), ("hi",)], ["comment_text"]))).collect()
assert probe_chars[0]["char_tokens"] == ["abc", "bcd", "cde", "abcd", "bcde", "abcde"]
assert probe_chars[1]["char_tokens"] == []

def char_stages(output="features"):
    return [
        lower_text.copy(), char_tokens.copy(),
        CountVectorizer(inputCol="char_tokens", outputCol="char_tf",
                        vocabSize=80000, minDF=2.0),
        IDF(inputCol="char_tf", outputCol="char_idf", minDocFreq=2),
        Normalizer(inputCol="char_idf", outputCol=output, p=2.0),
    ]

def feature_pipeline(representation):
    if representation == "baseline_words":
        return Pipeline(stages=[
            tokenizer.copy(),
            HashingTF(inputCol="word_tokens", outputCol="word_tf", numFeatures=1 << 17),
            IDF(inputCol="word_tf", outputCol="features", minDocFreq=2),
        ])
    if representation == "char35":
        return Pipeline(stages=char_stages())
    return Pipeline(stages=[
        tokenizer.copy(),
        NGram(n=2, inputCol="word_tokens", outputCol="bigrams"),
        SQLTransformer(statement="SELECT *, concat(word_tokens, bigrams) AS word12 FROM __THIS__"),
        CountVectorizer(inputCol="word12", outputCol="word_tf", vocabSize=50000, minDF=2.0),
        IDF(inputCol="word_tf", outputCol="word_idf", minDocFreq=2),
        Normalizer(inputCol="word_idf", outputCol="word_norm", p=2.0),
        *char_stages(output="char_norm"),
        VectorAssembler(inputCols=["word_norm", "char_norm"], outputCol="combined"),
        Normalizer(inputCol="combined", outputCol="features", p=2.0),
    ])

def measure(predictions):
    matrix = [[0] * 3 for _ in range(3)]
    for row in predictions.groupBy("label", "prediction").count().collect():
        truth, guess = int(row["label"]), int(row["prediction"])
        assert truth in range(3) and guess in range(3)
        matrix[truth][guess] += row["count"]
    n = sum(map(sum, matrix))
    assert n > 0
    per_class = {}
    for label, name in enumerate(LABEL_NAMES):
        tp = matrix[label][label]
        support = sum(matrix[label])
        predicted = sum(row[label] for row in matrix)
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * tp / (support + predicted) if support + predicted else 0.0
        per_class[name] = dict(support=support, precision=precision, recall=recall, f1=f1)
    return {
        "rows": n, "accuracy": sum(matrix[i][i] for i in range(3)) / n,
        "macro_f1": sum(v["f1"] for v in per_class.values()) / 3,
        "weighted_f1": sum(v["f1"] * v["support"] for v in per_class.values()) / n,
        "per_class": per_class, "confusion_matrix": matrix,
        "confusion_matrix_order": LABEL_NAMES,
    }

check = measure(spark.createDataFrame([(0.0, 0.0), (1.0, 1.0), (2.0, 2.0)],
                                    ["label", "prediction"]))
assert check["macro_f1"] == 1.0

# Grille fixée avant toute nouvelle évaluation du test.
GRID = [
    dict(name="baseline_words", representation="baseline_words", regParam=0.1,
         standardization=True, weighted=True, maxIter=30),
]
for representation in ("char35", "word12_char35"):
    for strength in (0.001, 0.01, 0.05):
        GRID.append(dict(name=representation + "_reg" + str(strength),
                         representation=representation, regParam=strength,
                         standardization=False, weighted=True, maxIter=80))
# Un témoin sans pondération mesure son utilité sur ce corpus déjà assez équilibré.
GRID.append(dict(name="word12_char35_unweighted", representation="word12_char35",
                 regParam=0.01, standardization=False, weighted=False, maxIter=80))
report["grid"] = GRID
report["feature_settings"] = {"char_n": [3, 4, 5], "word_n": [1, 2], "min_df": 2,
                              "char_vocab_limit": 80000, "word_vocab_limit": 50000,
                              "normalization": "L2 except baseline", "extra_cleaning": "lowercase"}


## 3. Comparer sur dev uniquement

L'IDF et le vocabulaire sont appris une fois par représentation, sur train. Les caractéristiques sont réutilisées pour les différentes valeurs de régularisation. Chaque candidat produit sa matrice de confusion et son F1 par classe. Le meilleur score dev détermine le modèle retenu.


In [ ]:
best_model, best_result = None, None
for representation in ("baseline_words", "char35", "word12_char35"):
    print("FEATURES", representation, flush=True)
    feature_model = feature_pipeline(representation).fit(train)
    train_vectors = (feature_model.transform(train).select("features", "label", "class_weight")
                     .repartition(4).persist(StorageLevel.MEMORY_AND_DISK))
    dev_vectors = (feature_model.transform(dev).select("features", "label")
                   .repartition(4).persist(StorageLevel.MEMORY_AND_DISK))
    assert train_vectors.count() == input_audit["train"]["rows"]
    assert dev_vectors.count() == input_audit["dev"]["rows"]
    for setting in [s for s in GRID if s["representation"] == representation]:
        started = time.monotonic()
        parameters = {k: setting[k] for k in ("regParam", "standardization", "maxIter")}
        if setting["weighted"]:
            parameters["weightCol"] = "class_weight"
        classifier = LogisticRegression(family="multinomial", tol=1e-5, **parameters)
        fitted = classifier.fit(train_vectors)
        metrics = measure(fitted.transform(dev_vectors))
        result = dict(setting, dev=metrics, elapsed_seconds=round(time.monotonic() - started, 2),
                      iterations=int(fitted.summary.totalIterations),
                      reached_max_iterations=int(fitted.summary.totalIterations) >= setting["maxIter"])
        report["results"].append(result)
        if best_result is None or metrics["macro_f1"] > best_result["dev"]["macro_f1"]:
            best_result = result
            best_model = PipelineModel(stages=feature_model.stages + [fitted])
        write_local("report.json", report)
        print("CANDIDATE", json.dumps(result, ensure_ascii=False), flush=True)
    train_vectors.unpersist()
    dev_vectors.unpersist()

assert best_model is not None
report["selected_model"] = best_result["name"]
report["selected_model_path"] = HDFS_DIR + "/model"
report["selected_on"] = "dev.macro_f1 only"
print("SELECTED", best_result["name"], best_result["dev"]["macro_f1"], flush=True)


## 4. Sauvegarder et vérifier le modèle retenu

Le modèle utilise un nouveau répertoire HDFS pour préserver les expériences précédentes. On recharge ensuite le modèle sauvegardé et on vérifie que ses prédictions sur dev sont identiques. Le test est évalué une fois pour le modèle choisi. Des sorties locales permettent de lire le rapport sans Docker.


In [ ]:
selected_dev = best_model.transform(dev).select("record_id", "comment_text", "label", "prediction").cache()
report["selected_train"] = measure(best_model.transform(train))
best_model.write().save(report["selected_model_path"])

reloaded = PipelineModel.load(report["selected_model_path"])
check_reload = reloaded.transform(dev).select("record_id", F.col("prediction").alias("reloaded_prediction"))
mismatches = (selected_dev.join(check_reload, "record_id")
              .where(F.col("prediction") != F.col("reloaded_prediction")).count())
assert mismatches == 0
report["reload_prediction_mismatches"] = mismatches

# Aucun score test n'a participé au choix du candidat ci-dessus.
test_predictions = (reloaded.transform(frames["test"])
                    .select("record_id", "comment_text", "label", "prediction").cache())
report["selected_test"] = measure(test_predictions)
assert report["selected_test"]["rows"] == input_audit["test"]["rows"]

for split, predictions in (("dev", selected_dev), ("test", test_predictions)):
    predictions.write.mode("errorifexists").parquet(HDFS_DIR + "/predictions/" + split)
    selected = predictions.select("record_id", "label", "prediction").orderBy("record_id").collect()
    with (LOCAL_DIR / (split + "_predictions.csv")).open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["record_id", "label", "prediction"])
        writer.writeheader()
        writer.writerows(r.asDict() for r in selected)

errors = (selected_dev.where(F.col("label") != F.col("prediction"))
          .orderBy("record_id").limit(25).collect())
write_local("dev_errors.json", [row.asDict() for row in errors])
report["status"] = "completed"
report["completed_at"] = datetime.now(timezone.utc).isoformat()
report["local_report_dir"] = str(LOCAL_DIR)
report["hdfs_run_dir"] = HDFS_DIR
write_local("report.json", report)
spark.createDataFrame([(json.dumps(report, ensure_ascii=False, allow_nan=False),)], ["value"]).coalesce(1).write.text(HDFS_DIR + "/metrics")
print("FINAL_REPORT", json.dumps(report, ensure_ascii=False, allow_nan=False), flush=True)
selected_dev.unpersist()
test_predictions.unpersist()
spark.stop()
